In [1]:
import pandas as pd

In [ ]:
from tqdm.notebook import tqdm
import json
def load_niat_dataset( 
    dataset_path,
    tag="test",
    first_n=-1,
):
    """Load niat JSONL (latest format) and normalize fields for CoT pipeline.

    - Adds incremental id: f"{tag}-{i}"
    - Initializes empty chain list per sample
    - Builds `table_text` from `table.header` + `table.rows` when available
    - Sets `table_caption` from `table.page_title` when available
    - Ensures both `question` and `statement` exist for compatibility
    - Respects `first_n` to limit number of loaded lines
    """
    dataset = []
    if first_n != -1:
        all_lines = []
        for line in open(dataset_path):
            all_lines.append(line)
            if len(all_lines) >= first_n:
                break
    else:
        all_lines = open(dataset_path).readlines()

    for i, line in tqdm(enumerate(all_lines), total=len(all_lines), desc=f"Loading niat-{tag} dataset"):
        info = json.loads(line)

        # Basic identifiers
        info["id"] = f"{tag}-{i}"
        info["chain"] = []

        # Derive table_text if not present
        if "table_text" not in info:
            info['table_text'] = info.get('table_rows', [])

        # Derive table_caption if not present
        if "table_caption" not in info:
            caption = info.get("table_tile", "")
            info["table_caption"] = caption

        dataset.append(info)
    return dataset

result = load_niat_dataset('../datasets/NIAT/sampled_qa_pairs_4000_fixed.json')

In [ ]:
result = load_niat_dataset('../')

Loading niat-test dataset:   0%|          | 0/1 [00:00<?, ?it/s]

TypeError: list indices must be integers or slices, not str

In [5]:
result = pd.read_csv('/data/workspace/yanmy/SPARQ/schedule_pipeline/tmp/niat_test/final_results.csv')

In [7]:
result.iloc[1,-1]

'[\'1. Decompose:\\n    - #1: Find the row where country = Denmark in the main table\\n    - #2: The place for Denmark in the main table is 12\\n    - #3: Verify with additional evidence: the additional evidence confirms that Denmark\\\'s place is 12\\n    - #4: Therefore, Denmark finished in position 12\\n\\n2. Final Answer: Therefore, the answer is: "12"\']'

In [17]:
raw = pd.read_json('/data/workspace/yanmy/SPARQ/datasets/NIAT/sampled_qa_pairs_4000_fixed.json')

In [19]:
raw.iloc[1]['table_rows']

[['', ' ', ' ', 'Class or Craft', 'Employees  (1)', 'Contract Amendable Date'],
 ['0',
  'Mainline: (2)',
  'Allied Pilots Association (APA)',
  'Pilots',
  '13,200',
  '2020'],
 ['1',
  '',
  'Association of Professional Flight Attendants (APFA)',
  'Flight Attendants',
  '24,900',
  '2019'],
 ['2',
  '',
  'Airline Customer Service Employee Association \\u2013Communications Workers of America and International Brotherhood of Teamsters (CWA-IBT)',
  'Passenger Service',
  '16,000',
  '2020'],
 ['3',
  '',
  'Transport Workers Union and International Association of Machinists & Aerospace Workers (TWU-IAM Association)',
  'Mechanics and Related',
  '12,400',
  '2018'],
 ['4', '', 'TWU-IAM Association', 'Fleet Service', '16,700', '2018'],
 ['5', '', '', 'Stock Clerks', '1,900', '2018'],
 ['6', '', '', 'Flight Simulator Engineers', '150', '2021'],
 ['7', '', '', 'Mainte   ce Control Technicians', '200', '2018'],
 ['8', '', '', 'Mainte   ce Training Instructors', '50', '2018'],
 ['9', '', 

In [12]:
import numpy as np
wikitq_data = np.load('/data/workspace/yanmy/SPARQ/schedule_pipeline/tmp/wikitq_test/wikitq_df_processed.npy',allow_pickle=True).item()

In [20]:
result.iloc[1]

table_id                                               AIT-QA_tab-37
dataset_name                                                  AIT-QA
table_structure                                         hierarchical
table_format                                                    HTML
table_title                                                      NaN
table_rows         [['', ' ', ' ', 'Class or Craft', 'Employees  ...
html_table         <table border="1" class="dataframe">\n  <thead...
question           How many Ground School Instructors are represe...
answer                                                            10
instruction        You are an expert on table data.\nYou must use...
predict            ['1. Decompose:\n    - #1: Find the row where ...
Name: 1, dtype: object

In [21]:
wikitq_data[1]

,description losses,1939/40,1940/41,1941/42,1942/43,1943/44,1944/45,total
0,direct war losses,360000,none,none,none,none,183000,543000
1,murdered,75000,100000,116000,133000,82000,none,506000
2,deaths in prisons & camps,69000,210000,220000,266000,381000,none,1146000
3,deaths outside of prisons & camps,none,42000,71000,142000,218000,none,473000
4,murdered in eastern regions,none,none,none,none,none,100000,100000
5,deaths other countries,none,none,none,none,none,none,2000
6,total,504000,352000,407000,541000,681000,270000,2770000
